# Etapa 2, E4: jediný záverečný beh na teste

Spustí **zmrazený** systém E4 (`E4_Qwen253BInstruct_E12kens_f030_prod`) na testovacích dátach. Všetky nastavenia boli vybrané len na valid (`docs/etapa2/zapisnik_experimentov.md`, časti E4) a po tomto behu sa už nemenia. Postup a dohoda s osobou 1 sú v `docs/etapa2/e4_test_beh.md`.

| Bunka | Čo robí | Čas |
| --- | --- | --- |
| 1 | zmrazené nastavenie, kontrola súborov | hneď |
| 2 | inštalácia torch a transformers, ak chýbajú | 0 až 5 min |
| 3 | kontrola revízie modelu (ak model chýba, stiahne presne túto revíziu) | hneď (sťahovanie asi 2 min) |
| 4 | kontrola pred testom bez GPU: valid z uložených odpovedí musí dať ST2 0,5171; kandidáti na teste | asi 10 s |
| 5 | kontrola GPU | pár sekúnd |
| 6 | **beh na teste** (997 správ, 1 994 otázok) | asi 12 až 15 min |
| 7 | ablácie na teste z uložených odpovedí (bez GPU) | asi 10 s |
| 8 | zbalenie výsledkov do `vysledky_e4_test.zip` | hneď |

Spúšťaj bunky postupne zhora (Shift+Enter). Ak bunka 1 alebo 4 skončí chybou, **nepokračuj** a pošli výpis. Ak sa bunka 6 preruší, spusti bunky 1 a 6 znova: odpovede sa ukladajú priebežne a beh pokračuje tam, kde skončil (výsledok bude rovnaký).

**Bunka 1:** zmrazené nastavenie. Nič v ňom nemeň (okrem `PRIECINOK_MODELOV`, ak je model inde). Všetky súbory musia byť `OK`.

In [ ]:
import glob, importlib.util, json, os, subprocess, sys, zipfile

# >>> zmrazené nastavenie E4 (vybrané na valid, nemeniť) <<<
PRIECINOK_MODELOV = "/home/jovyan/work/SpracovaniePrirodzenehoJazyka/zadanie_2etapa/llm_modely"
MODEL = "Qwen/Qwen2.5-3B-Instruct"
REVIZIA = "aa8e72537993ba99e69dfaafa59ed015b17504d1"   # commit modelu na Hugging Face
NASTAVENIE = ["--kandidati", "E12kens", "--k", "10", "--max-znakov", "1500", "--prompt", "v1",
              "--fuzia", "0.3", "--stitky-llm", "product"]
SYSTEM = "E4_Qwen253BInstruct_E12kens_f030_prod"

MODEL_CESTA = os.path.join(PRIECINOK_MODELOV, MODEL.split("/")[-1])
os.environ["HF_HOME"] = os.path.join(PRIECINOK_MODELOV, "hf_home")
os.environ["PIP_NO_CACHE_DIR"] = "1"
KOREN = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(KOREN)
print("koreň projektu:", KOREN, "| model:", MODEL_CESTA)
potrebne = ["src/spolocne.py", "src/e3_hierarchia.py", "src/e4_llm.py", "requirements_e4.txt",
            "data/incidents_valid.csv", "data/incidents_test.csv",
            "results/etapa2/kandidati/E12kens_valid.csv", "results/etapa2/predikcie/E12kens_valid.csv",
            "results/etapa2/kandidati/E12kens_test.csv", "results/etapa2/predikcie/E12kens_test.csv",
            "results/etapa2/llm_cache/Qwen_Qwen2.5-3B-Instruct.jsonl"]
for s in potrebne:
    print("OK    " if os.path.exists(s) else "CHÝBA ", s)
assert all(os.path.exists(s) for s in potrebne), "niektorý súbor chýba, skontroluj štruktúru priečinkov"

**Bunka 2:** doinštaluje torch a transformers, ak po reštarte servera chýbajú (bitsandbytes tu netreba).

In [ ]:
chybaju = [b for b in ("torch", "transformers") if importlib.util.find_spec(b) is None]
if chybaju:
    print("inštalujem:", chybaju)
    !{sys.executable} -m pip install --no-cache-dir -q torch==2.15.0 transformers==5.23.0
else:
    print("balíčky už sú nainštalované")

**Bunka 3:** overí, že stiahnutý model je presne revízia `REVIZIA`. Ak model v priečinku nie je, stiahne práve túto revíziu.

In [ ]:
meta = os.path.join(MODEL_CESTA, ".cache", "huggingface", "download", "config.json.metadata")
if not os.path.exists(meta):
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, revision=REVIZIA, local_dir=MODEL_CESTA,
                      allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*", "LICENSE*"])
with open(meta) as f:
    revizia = f.readline().strip()
print("revízia modelu:", revizia)
assert revizia == REVIZIA, "iná revízia modelu, než s akou sa ladilo na valid: zastav sa a pošli výpis"

**Bunka 4:** kontrola pred testom, beží bez GPU. (a) Hlavný systém na valid z uložených odpovedí musí dať presne ST2 0,5171 bez jediného volania modelu, teda kód, dáta, kandidáti aj odpovede sú tie isté ako pri ladení. (b) Kandidát č. 1 na teste musí byť predikcia E12kens a každá správa musí mať 10 kandidátov.

In [ ]:
import pandas as pd
sys.path.insert(0, "src")
from spolocne import nacitaj_kandidatov, nacitaj_predikcie

vystup = subprocess.run([sys.executable, "src/e4_llm.py", "--backend", "hf", "--model", MODEL,
                         "--model-cesta", MODEL_CESTA, "--split", "valid", *NASTAVENIE],
                        capture_output=True, text=True, encoding="utf-8")
print(vystup.stdout[-900:], vystup.stderr[-500:])
stat = json.loads(next(r for r in vystup.stdout.splitlines() if r.startswith("štatistika:")).split(":", 1)[1])
assert vystup.returncode == 0 and all(v["volania"] == 0 for v in stat.values()), "valid nešiel celý z cache"
skore_valid = json.load(open(f"results/etapa2/logy/{SYSTEM}_valid.json", encoding="utf-8"))["skore"]["ST2"]
assert skore_valid == 0.5171, f"ST2 na valid je {skore_valid}, čakali sme 0.5171"
print("(a) valid: ST2", skore_valid, "z cache, OK")

kand = nacitaj_kandidatov("E12kens", "test")
pred = nacitaj_predikcie("E12kens", "test").set_index("id")
for lab in ("hazard", "product"):
    g = kand[kand["stitok"] == lab]
    prvy = g[g["poradie"] == 1].set_index("id")["trieda"]
    assert len(prvy) == 997 and (prvy == pred[lab].reindex(prvy.index)).all(), f"{lab}: kandidát 1 ≠ predikcia"
    assert (g.groupby("id").size() == 10).all() and not g["skore"].isna().any(), f"{lab}: chybní kandidáti"
print("(b) test: 997 správ, kandidát 1 = predikcia E12kens, 10 kandidátov so skóre, OK")

**Bunka 5:** kontrola GPU.

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "torch nevidí GPU: je zvolená konfigurácia s GPU?"
print("GPU:", torch.cuda.get_device_name(0), f"| voľná pamäť {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

**Bunka 6: jediný beh na teste.** Model odpovedá pri hazarde aj pri produkte (1 994 otázok), aby sa z uložených odpovedí dali bez GPU spočítať aj ablácie v bunke 7; do výsledku hlavného systému ide LLM len pri produkte. Na konci vypíše skóre na teste oproti E12kens. **Po tomto behu sa nastavenie E4 už nemení**, ani keby výsledok bol iný, ako sme čakali.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --split test {" ".join(NASTAVENIE)} --povolit-test

**Bunka 7:** ablácie na teste z uložených odpovedí, bez GPU (nastavenia tiež z valid): samotný LLM a spojenie pri oboch štítkoch s w = 0,1. Musia vypísať `"volania": 0`.

In [ ]:
ZAKLAD = ["--backend", "hf", "--model", MODEL, "--model-cesta", MODEL_CESTA, "--split", "test",
          "--kandidati", "E12kens", "--k", "10", "--max-znakov", "1500", "--prompt", "v1", "--povolit-test"]
for navyse in ([], ["--fuzia", "0.1"]):
    print("=====", " ".join(navyse) or "LLM sám")
    v = subprocess.run([sys.executable, "src/e4_llm.py", *ZAKLAD, *navyse], capture_output=True, text=True, encoding="utf-8")
    print(v.stdout[-1200:], v.stderr[-500:])

**Bunka 8:** zbalí výsledky do `vysledky_e4_test.zip` v koreni projektu (aj s uloženými odpoveďami). Stiahni ho a vlož do koreňa projektu na svojom PC.

In [ ]:
subory = sorted(["results/etapa2/llm_cache/Qwen_Qwen2.5-3B-Instruct.jsonl"]
                + glob.glob("results/etapa2/predikcie/E4_*_test.csv") + glob.glob("results/etapa2/llm/detail_E4_*_test.csv")
                + glob.glob("results/etapa2/logy/E4_*_test.json"))
with zipfile.ZipFile("vysledky_e4_test.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for s in subory:
        z.write(s)
        print(s)
print(f"\n{len(subory)} súborov v {os.path.join(KOREN, 'vysledky_e4_test.zip')}")